# XGBoost Receipt Parser: Validation Evaluation

This notebook loads the latest saved XGBoost model, evaluates it on the local `data/validation` set, visualizes a probe image with the **predicted total token** emphasized, and shows a SHAP explanation for that prediction.

Assumptions:
- `data/validation` follows the SROIE-like folder structure (`img/`, `box/`, `entities/`).
- The model is saved under `models/xgb_parser` (best model in `archive/` or `best.ubj`).

In [ ]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

# Make sure we can import project utils
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from utils import load_ocr, load_entity_data, match_labels, add_features, fix_total_label, FEATURES as DEFAULT_FEATURES

In [ ]:
# ---- Configuration ----
VALIDATION_PATH = repo_root / "data" / "validation"
FEATURES = DEFAULT_FEATURES  # keep consistent with training

# Choose latest archived model by mtime; fallback to models/xgb_parser/best.ubj
MODELS_DIR = repo_root / "models" / "xgb_parser"

In [ ]:
def find_latest_model(models_dir: Path) -> Path:
    candidates = list(models_dir.glob("archive/*/best.ubj"))
    if candidates:
        return max(candidates, key=lambda p: p.stat().st_mtime)
    fallback = models_dir / "best.ubj"
    if fallback.exists():
        return fallback
    raise FileNotFoundError("No model found under models/xgb_parser")

model_path = find_latest_model(MODELS_DIR)
print(f"Using model: {model_path}")

In [ ]:
# Load model
model = xgb.XGBClassifier()
model.load_model(str(model_path))

In [ ]:
# ---- Load validation data ----
ocr_df = load_ocr(str(VALIDATION_PATH))
labels_df = load_entity_data(str(VALIDATION_PATH))

labeled_df = match_labels(ocr_df, labels_df)
features_df = add_features(labeled_df)
features_df = fix_total_label(features_df)

X_val = features_df[FEATURES]
y_val = features_df["label"].astype(int)

In [ ]:
# ---- Evaluation ----
val_probs = model.predict_proba(X_val)[:, 1]
val_pred = (val_probs >= 0.5).astype(int)

report = classification_report(y_val, val_pred, output_dict=True)
roc_auc = roc_auc_score(y_val, val_pred)
cm = confusion_matrix(y_val, val_pred)

print("ROC_AUC:", round(roc_auc, 4))
print(pd.DataFrame(report).T)

print("Confusion matrix (rows=true, cols=pred):")
print(cm)

## Probe Image Visualization

Pick a file to inspect. By default we pick the file that contains the **highest predicted probability** token.

In [ ]:
# Attach predictions back to the dataframe
features_df = features_df.copy()
features_df["pred_prob"] = val_probs
features_df["pred"] = val_pred

# Select a probe file
probe_file = (
    features_df.sort_values("pred_prob", ascending=False)
    .iloc[0]["file"]
)
print("Probe file:", probe_file)

In [ ]:
def render_prediction(df_file: pd.DataFrame, image_path: Path) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    # Identify the predicted total token (highest probability)
    pred_idx = df_file["pred_prob"].idxmax()
    pred_row = df_file.loc[pred_idx]

    for idx, row in df_file.iterrows():
        x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x3), int(row.y3)
        if idx == pred_idx:
            color = (255, 0, 0)  # red
            thickness = 3
        else:
            color = (0, 255, 0)  # green
            thickness = 1
        cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

    # Annotate predicted token text
    label = f"pred: {str(pred_row['text'])}"
    x_text = max(0, int(pred_row.x1))
    y_text = max(0, int(pred_row.y1) - 10)
    cv2.putText(img, label, (x_text, y_text), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    # Resize for display if very large
    max_w = 1000
    h, w = img.shape[:2]
    if w > max_w:
        scale = max_w / w
        img = cv2.resize(img, (int(w * scale), int(h * scale)))

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Render
file_df = features_df[features_df["file"] == probe_file]
img_path = VALIDATION_PATH / "img" / f"{probe_file}.jpg"
rendered = render_prediction(file_df, img_path)

plt.figure(figsize=(8, 10))
plt.imshow(rendered)
plt.axis("off")
plt.title(f"Predicted total token highlighted for {probe_file}")
plt.show()

## SHAP Explanation

We compute SHAP values for the predicted token in the probe image.

In [ ]:
# Build SHAP explanation for the predicted token
file_df = features_df[features_df["file"] == probe_file]
pred_idx = file_df["pred_prob"].idxmax()

X_probe = file_df.loc[[pred_idx], FEATURES]

explainer = shap.TreeExplainer(model)
shap_values = explainer(X_probe)

print("Predicted probability:", round(file_df.loc[pred_idx, "pred_prob"], 4))
print("True label:", int(file_df.loc[pred_idx, "label"]))

shap.plots.waterfall(shap_values[0], max_display=12)

## Feature Name Mapping

This mapping lets you rename features for plots. By default it is identity, so you can edit later.

In [ ]:
# Feature name mapping (identity by default)
FEATURE_NAME_MAP = {name: name for name in FEATURES}

## SHAP Feature Importance on Test Dataset

We compute a SHAP summary plot on the **test dataset** to estimate global feature importance.

In [ ]:
# Load test dataset (SROIE-style)
TEST_PATH = repo_root / "data" / "SROIE2019" / "test"

ocr_df_test = load_ocr(str(TEST_PATH))
labels_df_test = load_entity_data(str(TEST_PATH))

labeled_df_test = match_labels(ocr_df_test, labels_df_test)
features_df_test = add_features(labeled_df_test)
features_df_test = fix_total_label(features_df_test)

with_total_files_test = features_df_test.loc[features_df_test["label"] == 1, "file"].unique()
features_df_test = features_df_test[features_df_test["file"].isin(with_total_files_test)].reset_index(drop=True)

X_test = features_df_test[FEATURES]

# Sample for speed if dataset is large
sample_size = min(2000, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=42) if len(X_test) > 0 else X_test

# Apply feature name mapping for plot display
X_test_sample_renamed = X_test_sample.rename(columns=FEATURE_NAME_MAP)

explainer_test = shap.TreeExplainer(model)
shap_values_test = explainer_test(X_test_sample_renamed)

shap.summary_plot(shap_values_test, X_test_sample_renamed, show=True)